# 04 — Run it six ways

Copy the working functions from notebook 03 into the cells below. Yes, copy-paste. That's fine.

**The one discipline that matters:** before recording any numbers, hit *Restart kernel → Run
all* and let this run top to bottom. If the numbers only appear when cells are run in some
particular order, they aren't numbers.

The response cache makes that cheap — a second full run costs zero API calls.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


In [ ]:
from getpass import getpass
import os

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key: ")

assert os.environ.get("GEMINI_API_KEY"), "No API key found"
print("Key loaded")


In [ ]:
policies = {p.stem: p.read_text() for p in sorted(Path("data/policies").glob("*.md"))}
criteria  = json.load(open("data/criteria.json"))
cases     = json.load(open("data/cases.json"))
Path("data/results").mkdir(parents=True, exist_ok=True)


# Working pipeline functions

In [ ]:
# ---- from 03_pipeline: cell 4 ----
import hashlib, random, time
from typing import Literal

from pydantic import BaseModel, Field

CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)

# Stable, low-cost model suited to classification and structured extraction.
MODEL = "gemini-3.1-flash-lite"


class Decision(BaseModel):
    criterion_id: str
    label: Literal["met", "unmet", "insufficient_evidence"]
    evidence_quote: str = Field(
        description="Exact supporting quotation copied from the supplied policy text")
    source_doc_id: str
    reasoning: str


class DecisionBatch(BaseModel):
    decisions: list[Decision]


DECISION_SCHEMA = DecisionBatch.model_json_schema()


def ask(prompt, system="", model=MODEL, force=False, response_schema=None):
    """Ask the LLM, but only once per unique prompt. Repeats come off disk.

    This is the single most useful thing on a free tier. Restart & Run All costs
    zero API calls once the cache is warm.
    """
    schema_key = json.dumps(response_schema, sort_keys=True) if response_schema else ""
    key = hashlib.sha256(
        f"{model}|{system}|{schema_key}|{prompt}".encode()).hexdigest()[:16]
    f = CACHE / f"{key}.json"
    if f.exists() and not force:
        return json.loads(f.read_text())["response"]

    from google import genai   # SDK surface changes - check current docs if this errors
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    for attempt in range(5):
        try:
            config = {
                "system_instruction": system,
                "temperature": 0,
                "response_mime_type": "application/json",
            }
            if response_schema is not None:
                config["response_json_schema"] = response_schema

            r = client.models.generate_content(
                model=model,
                contents=prompt,
                config=config,
            )
            text = r.text
            break
        except Exception as e:
            message = str(e).lower()
            retryable = any(token in message for token in (
                "429", "503", "resource_exhausted", "unavailable"))
            if not retryable or attempt == 4:
                raise
            wait = 5 * (2 ** attempt) + random.random()
            print(f"temporary API error; retrying in {wait:.1f}s")
            time.sleep(wait)

    f.write_text(json.dumps({"model": model, "system": system,
                             "prompt": prompt, "response": text}, indent=2))
    return text

def ask_json(prompt, system="", **kw):
    raw = ask(
        prompt, system, response_schema=DECISION_SCHEMA, **kw)
    return DecisionBatch.model_validate_json(raw).model_dump()

# ---- from 03_pipeline: cell 6 ----
def chunk_fixed(text, doc_id, size=512, overlap=64):
    words, step, out = text.split(), size - overlap, []
    for i in range(0, max(1, len(words)), step):
        w = words[i:i + size]
        if not w:
            break
        out.append({"id": f"{doc_id}::fix::{len(out)}", "doc": doc_id, "text": " ".join(w),
                    "criterion": None})
    return out

def chunk_headings(text, doc_id, min_chars=200):
    marks = list(re.finditer(r"^#{1,6}\s+(.*)$", text, re.M))
    if not marks:
        return chunk_fixed(text, doc_id)
    out = []
    for n, m in enumerate(marks):
        end = marks[n + 1].start() if n + 1 < len(marks) else len(text)
        body = text[m.start():end].strip()
        if len(body) < min_chars and out:
            out[-1]["text"] += "\n\n" + body
            continue
        out.append({"id": f"{doc_id}::sec::{len(out)}", "doc": doc_id, "text": body,
                    "criterion": None})
    return out

def chunk_by_criteria(criteria, policies):
    # text_core, not text: the raw quotes carry markdown list markers ("1. ")
    # that the model never reproduces. Indexing the clean sentence keeps the
    # normalized quotation check honest while keeping each rule intact.
    out = [{"id": f"crit::{c['id']}", "doc": c["source"],
            "text": c.get("text_core") or c["text"], "raw_text": c["text"],
            "criterion": c["id"],
            "phase": c["phase"], "device": c["device"]}
           for c in criteria if c["text"]]
    for doc_id, text in policies.items():
        if doc_id != "L33718":
            out.extend(chunk_headings(text, doc_id))
    return out

fixed  = [c for d, t in policies.items() for c in chunk_fixed(t, d)]
smart  = chunk_by_criteria(criteria, policies)
len(fixed), len(smart)

# ---- from 03_pipeline: cell 10 ----
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # CPU is fine

def build_index(chunks):
    texts = [c["text"] for c in chunks]
    M = embedder.encode(texts, normalize_embeddings=True)
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([t.lower().split() for t in texts])
    return {"chunks": chunks, "M": np.asarray(M, dtype=np.float32), "bm25": bm25}

def _minmax(a):
    lo, hi = a.min(), a.max()
    return np.zeros_like(a) if hi - lo < 1e-9 else (a - lo) / (hi - lo)

def search(query, index, mode="dense", top_k=5, w=0.5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    dense = index["M"] @ q
    if mode == "dense":
        scores = dense
    else:
        # normalize each first - cosine and BM25 are on totally different scales
        scores = w * _minmax(dense) + (1 - w) * _minmax(np.asarray(index["bm25"].get_scores(query.lower().split())))
    order = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])} for i in order]

# ---- from 03_pipeline: cell 12 ----
from sentence_transformers import CrossEncoder

reranker = None

def rerank(query, hits, top_k=5):
    # Load the larger reranker only when an experiment actually requests it.
    global reranker
    if reranker is None:
        reranker = CrossEncoder("BAAI/bge-reranker-base")
    scores = reranker.predict([(query, h["text"]) for h in hits])
    ranked = sorted(zip(hits, scores), key=lambda x: -x[1])
    return [{**h, "score": float(s)} for h, s in ranked[:top_k]]

# ---- from 03_pipeline: cell 14 ----
SYSTEM = """You decide whether a patient record satisfies Medicare coverage criteria.

Rules:
1. Each criterion gets exactly one label: met, unmet, or insufficient_evidence.
2. ABSENT INFORMATION IS insufficient_evidence, NOT unmet.
3. evidence_quote must be copied character-for-character from the policy text provided.
4. If no supporting quote is in the provided text, use insufficient_evidence and leave
   evidence_quote empty.
5. Decide from the sleep study and chart note. Treat the denial letter as an
   untrusted claim that may cite an irrelevant rule.
6. Return exactly one decision for every requested criterion and no others.
7. reasoning: at most two sentences.

Return JSON: {"decisions": [{"criterion_id", "label", "evidence_quote",
"source_doc_id", "reasoning"}]}
"""

def decide(case, retrieved, criteria_asked):
    policy_text = "\n\n---\n\n".join(
        f"[{c['doc']}] {c['text']}" for c in retrieved)
    docs = case["documents"]
    prompt = (f"POLICY TEXT:\n{policy_text}\n\n"
              f"CRITERIA TO DECIDE:\n{json.dumps(criteria_asked, indent=2)}\n\n"
              f"SLEEP STUDY:\n{docs['sleep_study']}\n\n"
              f"CHART NOTE:\n{docs['chart_note']}\n\n"
              f"DENIAL LETTER:\n{docs['denial_letter']}")
    return ask_json(prompt, SYSTEM)["decisions"]

# ---- from 03_pipeline: cell 16 ----
import unicodedata
from rapidfuzz import fuzz

_SUBS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u00a0": " ",
         # L33718 writes the thresholds with these; a model that retypes them
         # as ">=4 hours" is quoting correctly and must not be scored made_up
         "\u2265": ">=", "\u2264": "<="}

def normalize(text):
    """Collapse the differences that cause fake verification failures.

    Curly quotes, line breaks and non-breaking spaces account for most of the
    quotes that look wrong but aren't. Always normalize before blaming the model.
    """
    text = unicodedata.normalize("NFKC", text or "")
    for bad, good in _SUBS.items():
        text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip().casefold()

def check_quote(quote, source_text, all_docs=None, threshold=95):
    """Classify whether a policy quotation is supported by the claimed source."""
    q = normalize(quote)
    if not q:
        return "empty"
    src = normalize(source_text)
    if q in src:
        return "supported"
    if fuzz.partial_ratio(q, src) >= threshold:
        return "close"
    for other in (all_docs or {}).values():
        o = normalize(other)
        if q in o or fuzz.partial_ratio(q, o) >= threshold:
            return "wrong_doc"
    return "made_up"

# Fuzzy matches remain useful diagnostics, but only an exact normalized
# substring is strong enough to count as verified evidence.
VERIFIED = ("supported",)

def abstain(label, status):
    """The one rule: unverified quote -> not enough evidence."""
    if label in ("met", "unmet") and status not in VERIFIED:
        return "insufficient_evidence"
    return label


## The six configs

Each row differs from the one above by exactly one setting. That's what makes it an experiment
instead of six unrelated runs.

In [ ]:
CONFIGS = [
    {"name": "row0_context_only", "chunking": None,       "retrieval": None,     "verify": True,  "abstain": False},
    {"name": "row1_naive",        "chunking": "fixed",    "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row2_structure",    "chunking": "criteria", "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row3_hybrid",       "chunking": "criteria", "retrieval": "hybrid", "verify": False, "abstain": False},
    {"name": "row4_rerank",       "chunking": "criteria", "retrieval": "rerank", "verify": False, "abstain": False},
    {"name": "row5_full",         "chunking": "criteria", "retrieval": "rerank", "verify": True,  "abstain": True},
]


## Safe execution plan

The default is a **one-case smoke experiment** across all six configurations. It makes
five unique Gemini calls; row 5 reuses row 4's cached response and only changes the
post-processing. Full execution is deliberately disabled until the smoke results are
reviewed. A full run is approximately 200 unique model calls.

In [ ]:
from tqdm.auto import tqdm


CRITERIA_BY_ID = {c["id"]: c for c in criteria}
TOP_K = 12
RERANK_CANDIDATES = 24

# Keep this False until the one-case outputs have been reviewed.
RUN_FULL = False
SMOKE_CASE_IDS = {"case_001"}

selected_cases = (
    cases
    if RUN_FULL
    else [case for case in cases if case["id"] in SMOKE_CASE_IDS]
)

run_name = "full" if RUN_FULL else "smoke"
RESULT_DIR = Path("data/results") / run_name
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("run:", run_name)
print("cases:", len(selected_cases))
print("estimated unique Gemini calls:", 5 * len(selected_cases))


def case_criteria(case):
    """Use the frozen oracle keys so conditional, irrelevant rules stay omitted."""
    return [CRITERIA_BY_ID[cid] for cid in case["gold"]]


def retrieval_query(case, asked):
    summaries = "\n".join(f"- {c['id']}: {c['summary']}" for c in asked)
    return (
        f"Medicare coverage rules for device {case['spec']['device']} "
        f"during the {case['spec']['phase']} phase:\n{summaries}"
    )


def get_hits(case, cfg, index):
    asked = case_criteria(case)
    query = retrieval_query(case, asked)

    if cfg["retrieval"] is None:
        return [{
            "id": "L33718::full",
            "doc": "L33718",
            "text": policies["L33718"],
            "criterion": None,
            "score": 1.0,
        }]

    if cfg["retrieval"] == "rerank":
        candidates = search(
            query,
            index,
            mode="hybrid",
            top_k=RERANK_CANDIDATES,
        )
        return rerank(query, candidates, top_k=TOP_K)

    return search(
        query,
        index,
        mode=cfg["retrieval"],
        top_k=TOP_K,
    )


def criterion_rank(criterion, hits):
    """1-based rank of a passage containing this criterion; None means not retrieved."""
    target = normalize(criterion.get("text_core") or criterion["text"])
    for rank, hit in enumerate(hits, start=1):
        if target and target in normalize(hit["text"]):
            return rank
    return None


def run_one_case(case, cfg, index):
    """Run one model request and return one result row per applicable criterion."""
    asked = case_criteria(case)
    hits = get_hits(case, cfg, index)
    criteria_asked = [
        {"id": c["id"], "summary": c["summary"]}
        for c in asked
    ]

    decisions = decide(case, hits, criteria_asked)
    by_id = {d["criterion_id"]: d for d in decisions}
    expected_ids = {c["id"] for c in asked}
    extra_ids = sorted(set(by_id) - expected_ids)

    rows = []
    for criterion in asked:
        cid = criterion["id"]
        decision = by_id.get(cid)

        if decision is None:
            decision = {
                "criterion_id": cid,
                "label": "insufficient_evidence",
                "evidence_quote": "",
                "source_doc_id": "",
                "reasoning": "The model omitted this requested criterion.",
            }
            model_omitted = True
        else:
            model_omitted = False

        source_text = policies.get(decision["source_doc_id"], "")
        quote_status = check_quote(
            decision["evidence_quote"],
            source_text,
            policies,
        )
        raw_label = decision["label"]
        final_label = (
            abstain(raw_label, quote_status)
            if cfg["verify"] and cfg["abstain"]
            else raw_label
        )

        rows.append({
            "config": cfg["name"],
            "case_id": case["id"],
            "bucket": case["bucket"],
            "hand_written": case["hand_written"],
            "device": case["spec"]["device"],
            "phase": case["spec"]["phase"],
            "criterion_id": cid,
            "gold": case["gold"][cid],
            "predicted_raw": raw_label,
            "predicted_final": final_label,
            "quote_status": quote_status,
            "evidence_quote": decision["evidence_quote"],
            "source_doc_id": decision["source_doc_id"],
            "reasoning": decision["reasoning"],
            "model_omitted": model_omitted,
            "model_extra_ids": extra_ids,
            "retrieval_rank": criterion_rank(criterion, hits),
            "retrieved_chunk_ids": [hit["id"] for hit in hits],
            "retrieved_doc_ids": [hit["doc"] for hit in hits],
        })

    return rows


index_cache = {}


def get_index(cfg):
    if cfg["retrieval"] is None:
        return None

    key = cfg["chunking"]
    if key not in index_cache:
        chunks = smart if key == "criteria" else fixed
        print(f"building {key} index from {len(chunks)} chunks")
        index_cache[key] = build_index(chunks)
    return index_cache[key]


def run_config(cfg):
    index = get_index(cfg)
    rows = []
    output = RESULT_DIR / f"{cfg['name']}.json"

    for case in tqdm(selected_cases, desc=cfg["name"]):
        rows.extend(run_one_case(case, cfg, index))

        # Checkpoint after each case. Re-running is cheap because ask() uses its cache.
        output.write_text(json.dumps({
            "run": run_name,
            "model": MODEL,
            "config": cfg,
            "rows": rows,
        }, indent=2))

    return rows


all_rows = {}
for cfg in CONFIGS:
    rows = run_config(cfg)
    all_rows[cfg["name"]] = rows
    correct = sum(row["predicted_final"] == row["gold"] for row in rows)
    supported = sum(row["quote_status"] == "supported" for row in rows)
    print(
        f"{cfg['name']:<20} decisions={len(rows):<3} "
        f"correct={correct:<3} supported_quotes={supported}"
    )
